# Product-Qualified Account Scoring

## Senior Data Science Task

Build a leakage-aware machine learning system that predicts which accounts are most likely to become **Won / Closed Won** opportunities in the next 90 days.

This notebook uses the SaaS seed data in:

`lightdash-demo-saas/seeds`

The task is account-level prioritization for Sales, Growth, and Product teams.

### Business Question

Which accounts should the team prioritize this week because their product usage, marketing source, sales activity, firmographics, and geography suggest high conversion potential?

### Prediction Target

For each account snapshot date:

```text
Target = 1 if the account creates a Won / Closed Won deal in the next 90 days
Target = 0 otherwise
```

### Senior-Level Requirements Covered

- Time-based train/validation/test split
- Leakage prevention with snapshot dates
- Product usage features from event telemetry
- Sales and marketing activity features
- Firmographic and geography features
- Multiple models
- Probability calibration using a validation period
- Per-model evaluation metrics
- Precision@K, Recall@K, lift, and revenue capture
- Calibration tables
- Ranked account scoring output

### Important Data Limitation

`deals_raw.csv` has `created_date` and current/final `stage`, but no explicit close date. For this exercise, we use `created_date` as the opportunity date. In production, replace this with true opportunity close date or stage transition timestamp.

In [7]:
from pathlib import Path
import os

import pandas as pd
import snowflake.connector


def load_env(path: str = ".env") -> None:
    env_path = Path(path)

    if not env_path.exists():
        raise FileNotFoundError(f"Could not find {path}")

    for line in env_path.read_text().splitlines():
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(
            key.strip(),
            value.strip().strip('"').strip("'")
        )


load_env()

In [8]:
private_key_file='/path/to/rsa_key.p8'
connection_params = {
    "private_key_file": "/Users/naghmeh6/agentic_ai/ABtesting/rsa_key.p8",
    "private_key_file": "/Users/naghmeh6/agentic_ai/ABtesting/rsa_key.p8",
    "authenticator": os.environ["SNOWFLAKE_AUTHENTICATOR"],
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USERNAME"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

connection_params = {
    key: value
    for key, value in connection_params.items()
    if value is not None
}

In [10]:
def read_snowflake_table(table_name: str, date_cols: list[str] | None = None) -> pd.DataFrame:
    query = f"""
        SELECT *
        FROM {table_name}
    """

    with snowflake.connector.connect(**connection_params) as conn:
        df = pd.read_sql(query, conn)
    if not df.empty and df.columns[:1].values[0] == "C1":
        # Set the first row as the column headers
        df.columns = df.iloc[0].astype(str)
        # Drop the first row since it is now the header
        df = df[1:].reset_index(drop=True)
    
    df.columns = df.columns.str.lower()
    for col in date_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df
    

## 1. Load Data

We load all available seed tables, then parse dates and normalize IDs. The model will mainly use account, user, product event, deal, marketing lead, sales activity, and geography tables.

In [11]:
accounts = read_snowflake_table("ACCOUNTS_RAW")
users = read_snowflake_table("USERS_RAW",["created_at", "first_logged_in_at", "latest_logged_in_at"])
tracks = read_snowflake_table("TRACKS_RAW",["event_timestamp"])
deals = read_snowflake_table("DEALS_RAW",["created_date"])
user_deals = read_snowflake_table("USER_DEALS_RAW")

marketing_leads = read_snowflake_table("MARKETING_LEADS",["created_at", "converted_at"])
activities = read_snowflake_table("ACTIVITIES_RAW",["activity_timestamp"])
lead_geo = read_snowflake_table("LEAD_GEOGRAPHIC_DATA")
addresses = read_snowflake_table("ADDRESSES_RAW",["valid_from", "valid_to"])




/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_64371/3260262751.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_64371/3260262751.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_64371/3260262751.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipyker

In [12]:
# If needed, install dependencies in your notebook environment:
# %pip install pandas numpy scikit-learn matplotlib seaborn

from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SEED_DIR = Path("ABtesting/lightdash-demo-saas/seeds")
PREDICTION_HORIZON_DAYS = 90
OBSERVATION_WINDOWS = [7, 14, 30, 60, 90]
HIGH_INTENT_EVENTS = [
    "workspace_created",
    "report_generated",
    "dashboard_viewed",
    "workflow_created",
    "api_call_made",
    "report_exported",
]
WON_STAGES = {"won", "closed won"}

In [13]:
# Basic cleanup
accounts["estimated_annual_recurring_revenue"] = pd.to_numeric(
    accounts["estimated_annual_recurring_revenue"], errors="coerce"
)
users["experience_in_years"] = pd.to_numeric(users["experience_in_years"], errors="coerce")
users["is_marketing_opted_in"] = pd.to_numeric(users["is_marketing_opted_in"], errors="coerce").fillna(0)
deals["amount"] = pd.to_numeric(deals["amount"], errors="coerce").fillna(0)
deals["seats"] = pd.to_numeric(deals["seats"], errors="coerce")
marketing_leads["lead_cost"] = pd.to_numeric(marketing_leads["lead_cost"], errors="coerce").fillna(0)
marketing_leads["lead_id"] = pd.to_numeric(marketing_leads["lead_id"], errors="coerce").fillna(0)
activities["call_duration_seconds"] = pd.to_numeric(activities["call_duration_seconds"], errors="coerce")

deals["is_won"] = deals["stage"].str.lower().isin(WON_STAGES).astype(int)

# User/account mapping for product usage aggregation
tracks = tracks.merge(users[["user_id", "account_id", "first_logged_in_at"]], on="user_id", how="left")
marketing_leads = marketing_leads.merge(
    users[["user_id", "account_id"]], on="user_id", how="left", suffixes=("", "_from_user")
)
activities = activities.merge(
    marketing_leads[["lead_id", "user_id", "account_id"]], on="lead_id", how="left"
)

print("Won stage rate in deals:", deals["is_won"].mean().round(3))
print("Top product events:")
print(tracks["event_name"].value_counts().head(12))

Won stage rate in deals: 0.33
Top product events:
event_name
api_call_made         21466
file_downloaded       15124
notification_sent      9584
workspace_created      8548
report_generated       7377
login_successful       7341
workflow_triggered     6918
message_sent           5981
report_exported        4861
dashboard_viewed       2075
file_uploaded          1286
workflow_created       1264
Name: count, dtype: int64


## 2. Define Leakage-Aware Snapshot Dates

A row in the modeling dataset is an `account_id` at a `snapshot_date`.

Features only use information available **on or before** the snapshot date.

The target looks forward 90 days from that snapshot date.

We exclude accounts that already had a won deal before the snapshot because the business question is prioritizing accounts that have not yet converted.

In [14]:

def make_snapshot_dates() -> pd.DatetimeIndex:
    min_date = max(
        users["created_at"].min(),
        tracks["event_timestamp"].min(),
        marketing_leads["created_at"].min(),
    )
    max_date = deals["created_date"].max() - pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

    # Monthly snapshots keep the dataset compact and enforce temporal evaluation.
    dates = pd.date_range(
        min_date.normalize() + pd.offsets.MonthBegin(1),
        max_date.normalize(),
        freq="MS",
    )
    return dates

snapshot_dates = make_snapshot_dates()
snapshot_dates

DatetimeIndex(['2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01',
               '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01',
               '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01',
               '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01',
               '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01',
               '2025-03-01', '2025-04-01'],
              dtype='datetime64[ns]', freq='MS')

In [15]:
marketing_leads

,lead_id,user_id,deal_id,lead_source,campaign_name,utm_medium,lead_cost,lead_status,created_at,converted_at,sdr,industry,year,account_id
0,1,3fd124c9-849c-41ad-bd78-e7826cf36c70,56147971-2963-437a-8ae1-87351cf08c10,Organic,Q1 Paid Search,email,385.007354,new,2024-01-05 00:00:00.565990,NaT,Barbie,Cybersecurity,2024,1960b6da-a435-4ee7-9965-34a3ddc1743d
1,2,a2f9fd3b-2c80-45c5-8134-e01c65eb6da5,958b921a-4569-48c8-9947-ff6e968c1691,Organic,Customer Referral,paid_search,172.101999,contacted,2025-10-16 00:00:00.565990,NaT,Ken,DevOps,2025,e4f0d50f-954e-4e1f-98a0-c868a0dc6742
2,3,c75d22f2-a716-4163-8406-fcb61ca7c666,None,Facebook Ads,Summer Webinar,referral,408.579538,new,2023-11-10 00:00:00.565990,NaT,Ken,Managed IT Services,2023,8b834cc6-f6cd-4579-873d-b97149a79649
3,4,68c44733-bbf1-4486-beca-2251f3f198a0,None,Referral,Holiday Promo,paid_search,404.744453,qualified,2023-12-14 00:00:00.565990,NaT,Barbie,Cybersecurity,2023,512fd254-eaaa-4349-a2ce-5870b0bf0536
4,5,a25e3506-1438-490a-b4a2-b0fb95b01f6b,None,Facebook Ads,Summer Webinar,paid_search,318.239989,contacted,2024-01-12 00:00:00.565990,NaT,Midge,IT Consulting,2024,ef2d6a22-2a56-4a08-8828-db4f344fc21d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2863,2864,4b71ac10-8db3-41d5-ad8b-28cd86c19c60,None,Referral,Product Launch,social,456.421599,new,2025-06-03 00:00:00.565990,NaT,Jake,SaaS,2025,6489fc80-4cc2-4a02-b6d7-4d0904e21e0d
2864,2865,ff2cb7db-3b97-41e0-862e-c059a881c1a6,a397dfef-3e59-4c06-986f-e4b227133148,Email,SEO Blog,organic,420.665427,new,2025-08-26 00:00:00.565990,NaT,Midge,Cloud Computing,2025,d2aedc17-71f6-4220-b303-49ed62fe632d
2865,2866,11ad3d5a-3fdc-44c4-87cf-99bd86978eed,65c5d1cf-49ad-4bde-8281-610a34e324bf,Email,Spring Drip Campaign,organic,287.362068,new,2025-09-19 00:00:00.565990,NaT,Ken,DevOps,2025,eb316d13-88dc-4595-b053-c0edf0fa5140
2866,2867,3cb77258-b2fe-48fe-858a-309aa51e9c96,None,Organic,Summer Webinar,social,315.520841,new,2025-12-01 00:00:00.565990,NaT,Jake,SaaS,2025,dd084838-269c-4b32-a13f-919ca431865e


## 3. Feature Engineering

The feature set combines:

- Firmographics from `accounts_raw.csv`
- User base profile from `users_raw.csv`
- Product usage from `tracks_raw.csv`
- Marketing acquisition from `marketing_leads.csv`
- Sales activity from `activities_raw.csv`
- Pipeline context from `deals_raw.csv`
- Geography from `lead_geographic_data.csv`

This is intentionally account-level because Sales/Growth prioritization happens at the account level.

In [16]:
def safe_divide(numerator: float, denominator: float) -> float:
    if denominator in (0, 0.0) or pd.isna(denominator):
        return 0.0
    return numerator / denominator


def mode_or_unknown(series: pd.Series) -> str:
    series = series.dropna()
    if series.empty:
        return "Unknown"
    return str(series.mode().iloc[0])


def days_since(snapshot_date: pd.Timestamp, value: pd.Timestamp | None) -> float:
    if value is None or pd.isna(value):
        return np.nan
    return float((snapshot_date - value).days)


def build_features_for_snapshot(snapshot_date: pd.Timestamp) -> pd.DataFrame:
    horizon_end = snapshot_date + pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

    base = accounts.copy()
    base["snapshot_date"] = snapshot_date

    # -------------------------
    # Account/user features
    # -------------------------
    users_seen = users[users["created_at"] <= snapshot_date].copy()
    user_features = users_seen.groupby("account_id").agg(
        users_created=("user_id", "nunique"),
        users_first_logged_in=("first_logged_in_at", lambda s: s.le(snapshot_date).sum()),
        marketing_opt_in_rate=("is_marketing_opted_in", "mean"),
        avg_experience_years=("experience_in_years", "mean"),
        max_experience_years=("experience_in_years", "max"),
        primary_job_title=("job_title", mode_or_unknown),
    ).reset_index()
    base = base.merge(user_features, on="account_id", how="left")

    # -------------------------
    # Product usage features
    # -------------------------
    tracks_seen = tracks[
        (tracks["event_timestamp"] <= snapshot_date)
        & tracks["account_id"].notna()
    ].copy()
    
    if not tracks_seen.empty:
        usage_all = tracks_seen.groupby("account_id").agg(
            total_events_to_date=("event_id", "count"),
            active_users_to_date=("user_id", "nunique"),
            distinct_events_to_date=("event_name", "nunique"),
            last_event_at=("event_timestamp", "max"),
            first_event_at=("event_timestamp", "min"),
        ).reset_index()
        usage_all["days_since_last_event"] = usage_all["last_event_at"].apply(lambda x: days_since(snapshot_date, x))
        usage_all["days_since_first_event"] = usage_all["first_event_at"].apply(lambda x: days_since(snapshot_date, x))
        usage_all = usage_all.drop(columns=["last_event_at", "first_event_at"])
        base = base.merge(usage_all, on="account_id", how="left")

        for event_name in HIGH_INTENT_EVENTS:
            event_counts = (
                tracks_seen[tracks_seen["event_name"] == event_name]
                .groupby("account_id")
                .size()
                .rename(f"event_count__{event_name}")
                .reset_index()
            )
            base = base.merge(event_counts, on="account_id", how="left")
    
    for window in OBSERVATION_WINDOWS:
        start = snapshot_date - pd.Timedelta(days=window)
        window_tracks = tracks[
            (tracks["event_timestamp"] > start)
            & (tracks["event_timestamp"] <= snapshot_date)
            & tracks["account_id"].notna()
        ]
        window_features = window_tracks.groupby("account_id").agg(
            **{
                f"events_last_{window}d": ("event_id", "count"),
                f"active_users_last_{window}d": ("user_id", "nunique"),
                f"distinct_events_last_{window}d": ("event_name", "nunique"),
            }
        ).reset_index()
        base = base.merge(window_features, on="account_id", how="left")
    
    # Recent usage momentum: last 30 days vs previous 30 days.
    last_30 = tracks[
        (tracks["event_timestamp"] > snapshot_date - pd.Timedelta(days=30))
        & (tracks["event_timestamp"] <= snapshot_date)
        & tracks["account_id"].notna()
    ].groupby("account_id").size().rename("events_last_30d_raw")
    prev_30 = tracks[
        (tracks["event_timestamp"] > snapshot_date - pd.Timedelta(days=60))
        & (tracks["event_timestamp"] <= snapshot_date - pd.Timedelta(days=30))
        & tracks["account_id"].notna()
    ].groupby("account_id").size().rename("events_prev_30d_raw")
    momentum = pd.concat([last_30, prev_30], axis=1).fillna(0).reset_index()
    momentum["event_momentum_30d"] = momentum.apply(
        lambda row: safe_divide(row["events_last_30d_raw"] - row["events_prev_30d_raw"], row["events_prev_30d_raw"] + 1),
        axis=1,
    )
    base = base.merge(momentum[["account_id", "event_momentum_30d"]], on="account_id", how="left")

    # -------------------------
    # Marketing lead features
    # -------------------------
    leads_seen = marketing_leads[
        (marketing_leads["created_at"] <= snapshot_date)
        & marketing_leads["account_id"].notna()
    ].copy()
    if not leads_seen.empty:
        lead_features = leads_seen.groupby("account_id").agg(
            leads_to_date=("lead_id", "nunique"),
            total_lead_cost=("lead_cost", "sum"),
            avg_lead_cost=("lead_cost", "mean"),
            converted_leads_to_date=("converted_at", lambda s: s.notna().sum()),
            primary_lead_source=("lead_source", mode_or_unknown),
            primary_campaign=("campaign_name", mode_or_unknown),
            primary_utm_medium=("utm_medium", mode_or_unknown),
            primary_sdr=("sdr", mode_or_unknown),
        ).reset_index()
        base = base.merge(lead_features, on="account_id", how="left")

        for status in ["new", "contacted", "qualified", "converted", "disqualified"]:
            status_counts = (
                leads_seen[leads_seen["lead_status"].eq(status)]
                .groupby("account_id")
                .size()
                .rename(f"lead_status_count__{status}")
                .reset_index()
            )
            base = base.merge(status_counts, on="account_id", how="left")

    # -------------------------
    # Sales activity features
    # -------------------------
    activities_seen = activities[
        (activities["activity_timestamp"] <= snapshot_date)
        & activities["account_id"].notna()
    ].copy()
    if not activities_seen.empty:
        activity_features = activities_seen.groupby("account_id").agg(
            sales_activities_to_date=("activity_id", "count"),
            active_leads_touched=("lead_id", "nunique"),
            avg_call_duration_seconds=("call_duration_seconds", "mean"),
            last_activity_at=("activity_timestamp", "max"),
        ).reset_index()
        activity_features["days_since_last_sales_activity"] = activity_features["last_activity_at"].apply(
            lambda x: days_since(snapshot_date, x)
        )
        activity_features = activity_features.drop(columns=["last_activity_at"])
        base = base.merge(activity_features, on="account_id", how="left")

        for activity_type in activities_seen["activity_type"].dropna().unique():
            type_counts = (
                activities_seen[activities_seen["activity_type"].eq(activity_type)]
                .groupby("account_id")
                .size()
                .rename(f"activity_count__{activity_type}")
                .reset_index()
            )
            base = base.merge(type_counts, on="account_id", how="left")
    
    # -------------------------
    # Prior pipeline features
    # Avoid using final stage as a feature because stage may be updated after snapshot.
    # -------------------------
    prior_deals = deals[deals["created_date"] <= snapshot_date].copy()
    if not prior_deals.empty:
        prior_deal_features = prior_deals.groupby("account_id").agg(
            prior_deals_to_date=("deal_id", "nunique"),
            prior_pipeline_amount=("amount", "sum"),
            prior_avg_deal_amount=("amount", "mean"),
            prior_max_seats=("seats", "max"),
            primary_plan_to_date=("plan", mode_or_unknown),
        ).reset_index()
        base = base.merge(prior_deal_features, on="account_id", how="left")

    # -------------------------
    # Geography features from lead geography.
    # -------------------------
    lead_geo_seen = lead_geo.merge(
        marketing_leads[["lead_id", "account_id", "created_at"]], on="lead_id", how="left"
    )
    lead_geo_seen = lead_geo_seen[
        (lead_geo_seen["created_at"] <= snapshot_date)
        & lead_geo_seen["account_id"].notna()
    ]
    if not lead_geo_seen.empty:
        geo_features = lead_geo_seen.groupby("account_id").agg(
            primary_continent=("continent", mode_or_unknown),
            primary_country=("country_name", mode_or_unknown),
            distinct_countries=("country_code", "nunique"),
        ).reset_index()
        base = base.merge(geo_features, on="account_id", how="left")

    # -------------------------
    # Target construction
    # -------------------------
    already_won = deals[
        (deals["created_date"] <= snapshot_date)
        & deals["is_won"].eq(1)
    ].groupby("account_id").size().rename("won_before_snapshot").reset_index()

    future_won = deals[
        (deals["created_date"] > snapshot_date)
        & (deals["created_date"] <= horizon_end)
        & deals["is_won"].eq(1)
    ].groupby("account_id").agg(
        target_won_90d=("deal_id", "nunique"),
        target_revenue_90d=("amount", "sum"),
    ).reset_index()

    base = base.merge(already_won, on="account_id", how="left")
    base = base.merge(future_won, on="account_id", how="left")
    base["won_before_snapshot"] = base["won_before_snapshot"].fillna(0)
    base["target_won_90d"] = base["target_won_90d"].fillna(0)
    base["target_revenue_90d"] = base["target_revenue_90d"].fillna(0)
    base["target"] = (base["target_won_90d"] > 0).astype(int)

    # Exclude accounts already won before the snapshot. We are predicting net-new conversion.
    base = base[base["won_before_snapshot"].eq(0)].copy()

    return base

In [17]:
feature_frames = [build_features_for_snapshot(snapshot_date) for snapshot_date in snapshot_dates]
model_df = pd.concat(feature_frames, ignore_index=True)

# Fill count-like nulls with 0. Leave categorical nulls for imputation.
count_like_prefixes = (
    "users_", "event_", "events_", "active_", "distinct_", "lead_", "activity_", "sales_", "prior_"
)
for col in model_df.columns:
    if col.startswith(count_like_prefixes) or col in [
        "total_events_to_date",
        "active_users_to_date",
        "distinct_events_to_date",
        "total_lead_cost",
        "avg_lead_cost",
        "converted_leads_to_date",
        "event_momentum_30d",
        "days_since_last_event",
        "days_since_first_event",
        "days_since_last_sales_activity",
        "target_revenue_90d",
    ]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce").fillna(0)

print(model_df.shape)
print(model_df[["snapshot_date", "account_id", "target", "target_revenue_90d"]].head())
print("Target rate:", model_df["target"].mean().round(4))
print("Positive rows:", int(model_df["target"].sum()))

(18544, 74)
  snapshot_date                            account_id  target  \
0    2023-07-01  8f55adef-0abf-460f-9898-2cda579d9ba5       0   
1    2023-07-01  c5b92423-831e-457c-afdd-e517420672df       0   
2    2023-07-01  de9b63ea-04cc-43ec-b622-3969981f1d7c       0   
3    2023-07-01  a9867c25-fd83-4e1b-8551-3bff728b163d       0   
4    2023-07-01  61830633-16a3-49ad-9ff7-37a7d72f3aa8       0   

   target_revenue_90d  
0                 0.0  
1                 0.0  
2                 0.0  
3                 0.0  
4                 0.0  
Target rate: 0.0494
Positive rows: 916


## 4. Time-Based Train / Validation / Test Split

We split by `snapshot_date`, not randomly. This is critical because account behavior and conversion patterns drift over time.

- Train: earliest 60% of snapshot dates
- Validation: next 20%, used for probability calibration and model selection
- Test: final 20%, used only once for final reporting

In [18]:
unique_dates = np.array(sorted(model_df["snapshot_date"].unique()))
train_end = unique_dates[int(len(unique_dates) * 0.60)]
valid_end = unique_dates[int(len(unique_dates) * 0.80)]

train_mask = model_df["snapshot_date"] < train_end
valid_mask = (model_df["snapshot_date"] >= train_end) & (model_df["snapshot_date"] < valid_end)
test_mask = model_df["snapshot_date"] >= valid_end

train_df = model_df[train_mask].copy()
valid_df = model_df[valid_mask].copy()
test_df = model_df[test_mask].copy()

print("train", train_df.shape, train_df["snapshot_date"].min(), train_df["snapshot_date"].max(), train_df["target"].mean())
print("valid", valid_df.shape, valid_df["snapshot_date"].min(), valid_df["snapshot_date"].max(), valid_df["target"].mean())
print("test ", test_df.shape, test_df["snapshot_date"].min(), test_df["snapshot_date"].max(), test_df["target"].mean())

train (11810, 74) 2023-07-01 00:00:00 2024-07-01 00:00:00 0.04614733276883996
valid (3138, 74) 2024-08-01 00:00:00 2024-11-01 00:00:00 0.058317399617590825
test  (3596, 74) 2024-12-01 00:00:00 2025-04-01 00:00:00 0.05228031145717464


## 5. Preprocessing

We use numeric imputation/scaling and categorical one-hot encoding. The preprocessing is fit on train only.

In [19]:
TARGET_COL = "target"
REVENUE_COL = "target_revenue_90d"
DROP_COLS = [
    "account_id",
    "account_name",
    "snapshot_date",
    "target",
    "target_won_90d",
    "target_revenue_90d",
    "won_before_snapshot",
]

feature_cols = [c for c in model_df.columns if c not in DROP_COLS]
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]
X_valid = valid_df[feature_cols]
y_valid = valid_df[TARGET_COL]
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]
revenue_test = test_df[REVENUE_COL]

numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print("numeric", len(numeric_cols))
print("categorical", len(categorical_cols), categorical_cols[:10])


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

numeric 57
categorical 10 ['industry', 'segment', 'primary_job_title', 'primary_lead_source', 'primary_campaign', 'primary_utm_medium', 'primary_sdr', 'primary_plan_to_date', 'primary_continent', 'primary_country']


## 6. Evaluation Functions

For account prioritization, AUC alone is not enough. We care about the top-ranked accounts.

Metrics reported for every model:

- ROC-AUC
- PR-AUC
- Log loss
- Brier score
- Precision@10%
- Recall@10%
- Lift@10%
- Precision@50 accounts
- Recall@50 accounts
- Revenue captured@10%
- Revenue captured@50 accounts

In [20]:
def precision_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    if k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.mean(np.asarray(y_true)[order]))


def recall_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    positives = float(np.sum(y_true))
    if positives == 0 or k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.sum(np.asarray(y_true)[order]) / positives)


def revenue_capture_at_k(revenue: pd.Series, scores: np.ndarray, k: int) -> float:
    total_revenue = float(np.sum(revenue))
    if total_revenue == 0 or k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.sum(np.asarray(revenue)[order]) / total_revenue)


def evaluate_predictions(
    model_name: str,
    y_true: pd.Series,
    scores: np.ndarray,
    revenue: pd.Series,
    top_fraction: float = 0.10,
    top_n: int = 50,
) -> dict[str, Any]:
    y_true_arr = np.asarray(y_true)
    scores = np.clip(np.asarray(scores), 1e-6, 1 - 1e-6)
    k_fraction = max(1, int(len(y_true_arr) * top_fraction))
    k_n = min(top_n, len(y_true_arr))
    base_rate = float(np.mean(y_true_arr))

    return {
        "model": model_name,
        "rows": len(y_true_arr),
        "base_rate": base_rate,
        "roc_auc": roc_auc_score(y_true_arr, scores) if len(np.unique(y_true_arr)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true_arr, scores) if len(np.unique(y_true_arr)) > 1 else np.nan,
        "log_loss": log_loss(y_true_arr, scores, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true_arr, scores),
        "precision_at_10pct": precision_at_k(y_true, scores, k_fraction),
        "recall_at_10pct": recall_at_k(y_true, scores, k_fraction),
        "lift_at_10pct": safe_divide(precision_at_k(y_true, scores, k_fraction), base_rate),
        "revenue_capture_at_10pct": revenue_capture_at_k(revenue, scores, k_fraction),
        "precision_at_50": precision_at_k(y_true, scores, k_n),
        "recall_at_50": recall_at_k(y_true, scores, k_n),
        "lift_at_50": safe_divide(precision_at_k(y_true, scores, k_n), base_rate),
        "revenue_capture_at_50": revenue_capture_at_k(revenue, scores, k_n),
    }


def calibration_table(y_true: pd.Series, scores: np.ndarray, bins: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"y": y_true.values, "score": scores})
    df["score_bin"] = pd.qcut(df["score"], q=bins, duplicates="drop")
    return (
        df.groupby("score_bin", observed=True)
        .agg(
            accounts=("y", "size"),
            avg_predicted_probability=("score", "mean"),
            actual_win_rate=("y", "mean"),
            wins=("y", "sum"),
        )
        .reset_index()
    )

## 7. Probability Calibration

We fit each base model on the training period, then calibrate probabilities on the validation period using Platt scaling.

This avoids using the test period for calibration.

In [29]:
@dataclass
class FittedModel:
    name: str
    pipeline: Pipeline
    calibrator: LogisticRegression
    validation_metrics: dict[str, Any]
    test_metrics: dict[str, Any]
    valid_scores: np.ndarray
    test_scores: np.ndarray


def logit_transform(probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities)).reshape(-1, 1)


def fit_calibrated_model(name: str, estimator: Any) -> FittedModel:
    pipeline = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)

    raw_valid = pipeline.predict_proba(X_valid)[:, 1]
    calibrator = LogisticRegression(random_state=RANDOM_STATE)
    calibrator.fit(logit_transform(raw_valid), y_valid)

    valid_scores = calibrator.predict_proba(logit_transform(raw_valid))[:, 1]
    raw_test = pipeline.predict_proba(X_test)[:, 1]
    test_scores = calibrator.predict_proba(logit_transform(raw_test))[:, 1]

    return FittedModel(
        name=name,
        pipeline=pipeline,
        calibrator=calibrator,
        validation_metrics=evaluate_predictions(name, y_valid, valid_scores, valid_df[REVENUE_COL]),
        test_metrics=evaluate_predictions(name, y_test, test_scores, revenue_test),
        valid_scores=valid_scores,
        test_scores=test_scores,
    )

## 8. Train Multiple Models

Models included:

1. Logistic Regression: interpretable baseline
2. Random Forest: nonlinear interactions and robustness
3. Gradient Boosting: strong tabular baseline
4. Optional XGBoost: included if the package is installed

Every model is evaluated using the same metrics.

In [30]:
models: dict[str, Any] = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.04,
        max_depth=3,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
    ),
}

try:
    from xgboost import XGBClassifier

    models["xgboost"] = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )
except Exception as exc:
    print("XGBoost not available; skipping optional xgboost model.", exc)

fitted_models: list[FittedModel] = []
for name, estimator in models.items():
    print(f"Training {name}...")
    fitted_models.append(fit_calibrated_model(name, estimator))

validation_results = pd.DataFrame([m.validation_metrics for m in fitted_models]).sort_values(
    ["revenue_capture_at_10pct", "pr_auc"], ascending=False
)
test_results = pd.DataFrame([m.test_metrics for m in fitted_models]).sort_values(
    ["revenue_capture_at_10pct", "pr_auc"], ascending=False
)

print("Validation results")
display(validation_results)
print("Test results")
display(test_results)

Training logistic_regression...
Training random_forest...
Training gradient_boosting...
Training xgboost...
Validation results


,model,rows,base_rate,roc_auc,pr_auc,log_loss,brier_score,precision_at_10pct,recall_at_10pct,lift_at_10pct,revenue_capture_at_10pct,precision_at_50,recall_at_50,lift_at_50,revenue_capture_at_50
3,xgboost,3138,0.058317,0.833319,0.217085,0.175271,0.049686,0.217252,0.371585,3.725344,0.308489,0.40,0.109290,6.859016,0.088286
2,gradient_boosting,3138,0.058317,0.813926,0.177987,0.184633,0.051163,0.201278,0.344262,3.451422,0.300822,0.26,0.071038,4.458361,0.081641
1,random_forest,3138,0.058317,0.834619,0.223153,0.171716,0.049503,0.204473,0.349727,3.506206,0.284280,0.36,0.098361,6.173115,0.074950
0,logistic_regression,3138,0.058317,0.796085,0.139170,0.190593,0.052322,0.146965,0.251366,2.520086,0.241439,0.16,0.043716,2.743607,0.050230


Test results


,model,rows,base_rate,roc_auc,pr_auc,log_loss,brier_score,precision_at_10pct,recall_at_10pct,lift_at_10pct,revenue_capture_at_10pct,precision_at_50,recall_at_50,lift_at_50,revenue_capture_at_50
2,gradient_boosting,3596,0.05228,0.857479,0.179676,0.163338,0.046238,0.203343,0.388298,3.889468,0.437510,0.16,0.042553,3.060426,0.057374
1,random_forest,3596,0.05228,0.874532,0.200601,0.151051,0.044383,0.214485,0.409574,4.102590,0.397707,0.22,0.058511,4.208085,0.059999
3,xgboost,3596,0.05228,0.865420,0.187579,0.159473,0.045741,0.194986,0.372340,3.729627,0.391064,0.14,0.037234,2.677872,0.030374
0,logistic_regression,3596,0.05228,0.844179,0.180510,0.170135,0.046653,0.211699,0.404255,4.049310,0.360636,0.10,0.026596,1.912766,0.014785


## 9. Calibration Diagnostics

A calibrated model should have predicted probabilities that roughly match observed win rates.

For example, accounts scored around 0.30 should convert around 30% of the time.

In [31]:
best_model_name = validation_results.iloc[0]["model"]
best_model = next(model for model in fitted_models if model.name == best_model_name)

print("Best model selected by validation:", best_model.name)

calibration_valid = calibration_table(y_valid, best_model.valid_scores)
calibration_test = calibration_table(y_test, best_model.test_scores)

print("Validation calibration")
display(calibration_valid)
print("Test calibration")
display(calibration_test)

Best model selected by validation: xgboost
Validation calibration


,score_bin,accounts,avg_predicted_probability,actual_win_rate,wins
0,"(-0.00025600000000000004, 0.00228]",314,0.001756,0.000000,0
1,"(0.00228, 0.00321]",314,0.002730,0.000000,0
2,"(0.00321, 0.00452]",314,0.003810,0.003185,1
3,"(0.00452, 0.00652]",313,0.005413,0.000000,0
4,"(0.00652, 0.0114]",314,0.008334,0.000000,0
5,"(0.0114, 0.0509]",314,0.028541,0.044586,14
6,"(0.0509, 0.0818]",313,0.067905,0.102236,32
7,"(0.0818, 0.114]",314,0.096719,0.108280,34
8,"(0.114, 0.163]",314,0.136742,0.108280,34
9,"(0.163, 0.546]",314,0.231267,0.216561,68


Test calibration


,score_bin,accounts,avg_predicted_probability,actual_win_rate,wins
0,"(-0.000282, 0.00212]",360,0.001655,0.008333,3
1,"(0.00212, 0.00284]",360,0.002491,0.000000,0
2,"(0.00284, 0.00362]",359,0.003217,0.000000,0
3,"(0.00362, 0.00455]",360,0.004054,0.005556,2
4,"(0.00455, 0.00571]",359,0.005068,0.008357,3
5,"(0.00571, 0.00789]",360,0.006714,0.000000,0
6,"(0.00789, 0.0136]",359,0.010096,0.002786,1
7,"(0.0136, 0.0672]",360,0.033252,0.094444,34
8,"(0.0672, 0.137]",359,0.100923,0.208914,75
9,"(0.137, 0.685]",360,0.219540,0.194444,70


## 10. Precision / Recall Tradeoff

This helps choose an operating threshold if the team wants a fixed score cutoff instead of top-K account routing.

In [32]:
precision, recall, thresholds = precision_recall_curve(y_valid, best_model.valid_scores)
threshold_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})
threshold_table["f1"] = 2 * threshold_table["precision"] * threshold_table["recall"] / (
    threshold_table["precision"] + threshold_table["recall"]
).replace(0, np.nan)
threshold_table = threshold_table.sort_values("f1", ascending=False)
threshold_table.head(20)

,threshold,precision,recall,f1
2877,0.176261,0.253012,0.344262,0.291667
2876,0.176197,0.252000,0.344262,0.290993
2869,0.174016,0.249027,0.349727,0.290909
2889,0.180655,0.257384,0.333333,0.290476
2875,0.176035,0.250996,0.344262,0.290323
2868,0.173750,0.248062,0.349727,0.290249
2888,0.180653,0.256303,0.333333,0.289786
2874,0.175987,0.250000,0.344262,0.289655
2867,0.173385,0.247104,0.349727,0.289593
2860,0.171297,0.244361,0.355191,0.289532


## 11. Ranked Account Output

This is the deliverable Growth/Sales can use:

- Account score
- Predicted win probability
- Account metadata
- Actual test outcome for backtesting
- Future 90-day revenue for evaluation

In [33]:
account_scores = test_df[[
    "snapshot_date",
    "account_id",
    "account_name",
    "industry",
    "segment",
    "estimated_annual_recurring_revenue",
    "target",
    "target_revenue_90d",
]].copy()
account_scores["model"] = best_model.name
account_scores["predicted_win_probability_90d"] = best_model.test_scores
account_scores["priority_rank"] = account_scores["predicted_win_probability_90d"].rank(
    ascending=False, method="first"
).astype(int)
account_scores = account_scores.sort_values("predicted_win_probability_90d", ascending=False)

account_scores.head(30)

,snapshot_date,account_id,account_name,industry,segment,estimated_annual_recurring_revenue,target,target_revenue_90d,model,predicted_win_probability_90d,priority_rank
16117,2025-01-01,7f343b41-c85e-4c10-a7b6-359a1bbc97c1,"Town Sports International Holdings, Inc.",Education,SMB,400000,0,0.0,xgboost,0.684614,1
16840,2025-02-01,7f343b41-c85e-4c10-a7b6-359a1bbc97c1,"Town Sports International Holdings, Inc.",Education,SMB,400000,0,0.0,xgboost,0.569455,2
15377,2024-12-01,7f343b41-c85e-4c10-a7b6-359a1bbc97c1,"Town Sports International Holdings, Inc.",Education,SMB,400000,0,0.0,xgboost,0.567830,3
16260,2025-01-01,384d6410-a754-4b71-8e41-781036d5532c,Deltic Timber Corporation,Financial Services,Enterprise,5500000,0,0.0,xgboost,0.541559,4
15047,2024-12-01,c7e5dcb1-8400-4307-9e0a-1540142fe34e,"CommScope Holding Company, Inc.",Hospitality,SMB,475000,0,0.0,xgboost,0.526930,5
16890,2025-02-01,b9ad62b9-0eb7-4307-bcd1-dfc4ecd81ae9,"Alliance One International, Inc.",Healthcare,Midmarket,1300000,0,0.0,xgboost,0.505314,6
18254,2025-04-01,7f343b41-c85e-4c10-a7b6-359a1bbc97c1,"Town Sports International Holdings, Inc.",Education,SMB,400000,0,0.0,xgboost,0.493953,7
15793,2025-01-01,c7e5dcb1-8400-4307-9e0a-1540142fe34e,"CommScope Holding Company, Inc.",Hospitality,SMB,475000,0,0.0,xgboost,0.475289,8
15608,2024-12-01,f865c978-581b-4f42-a41c-64961db24156,"Apricus Biosciences, Inc.",Hospitality,SMB,125000,0,0.0,xgboost,0.447426,9
17552,2025-03-01,7f343b41-c85e-4c10-a7b6-359a1bbc97c1,"Town Sports International Holdings, Inc.",Education,SMB,400000,0,0.0,xgboost,0.433511,10


## 12. Feature Importance

For tree-based models, use model feature importances when available. For logistic regression, use absolute coefficient values.

For production, I would add SHAP explanations; this notebook keeps dependencies lighter and portable.

In [34]:
def get_feature_names(pipeline: Pipeline) -> list[str]:
    transformer = pipeline.named_steps["preprocess"]
    try:
        return transformer.get_feature_names_out().tolist()
    except Exception:
        return [f"feature_{i}" for i in range(transformer.transform(X_train.head(1)).shape[1])]


def get_feature_importance(fitted: FittedModel, top_n: int = 40) -> pd.DataFrame:
    model = fitted.pipeline.named_steps["model"]
    feature_names = get_feature_names(fitted.pipeline)

    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_[0])
    else:
        return pd.DataFrame({"message": ["Model does not expose native feature importance."]})

    return (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

feature_importance = get_feature_importance(best_model)
feature_importance

,feature,importance
0,num__prior_deals_to_date,0.039486
1,num__prior_pipeline_amount,0.017593
2,cat__primary_sdr_Midge,0.016750
3,num__prior_avg_deal_amount,0.016555
4,cat__primary_job_title_DevOps Engineer,0.015562
5,cat__industry_Banking,0.013335
6,cat__primary_country_Australia,0.012879
7,num__event_count__workspace_created,0.012852
8,cat__primary_utm_medium_paid_search,0.012440
9,num__activity_count__follow_up,0.012317


## 13. Save Outputs and Publish to Snowflake

We keep local CSV copies as artifacts, then publish the scored results to Snowflake tables.
These tables (`ML_MODEL_METRICS`, `ML_CALIBRATION`, `ML_FEATURE_IMPORTANCE`, `ML_ACCOUNT_SCORES`)
are the production source of truth that the app reads through its API. The app never reads a CSV.

Auth uses the **RSA key pair** (`ABtesting/rsa_key.p8`) — the exact same key-pair (JWT) login the
app uses on its server. No browser popup or MFA. Requires
`pip install "snowflake-connector-python[pandas]" cryptography`.

In [35]:
import os

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from snowflake.connector.pandas_tools import write_pandas


def load_env(path: str = "ABtesting/.env") -> None:
    """Load Snowflake credentials from the project .env file."""
    env_path = Path(path)
    if not env_path.exists():
        print(f"Warning: {env_path} not found; relying on existing environment variables.")
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
    "xgboost": "XGBoost",
}


def build_metrics_frame() -> pd.DataFrame:
    frames = []
    for dataset, df in [("validation", validation_results), ("test", test_results)]:
        part = df.copy()
        part["dataset"] = dataset
        frames.append(part)
    metrics = pd.concat(frames, ignore_index=True)
    metrics["model_label"] = metrics["model"].map(MODEL_LABELS).fillna(metrics["model"])
    metrics["is_deployed"] = metrics["model"] == best_model.name
    metrics = metrics.rename(columns={"model": "model_name"})
    metrics.columns = [c.upper() for c in metrics.columns]
    return metrics


def build_calibration_frame() -> pd.DataFrame:
    frames = []
    for dataset, df in [("validation", calibration_valid), ("test", calibration_test)]:
        part = df.copy().reset_index(drop=True)
        part["dataset"] = dataset
        part["bin_order"] = range(len(part))
        part["score_bin"] = part["score_bin"].astype(str)
        frames.append(part)
    calibration = pd.concat(frames, ignore_index=True)
    calibration.columns = [c.upper() for c in calibration.columns]
    return calibration


def build_feature_importance_frame() -> pd.DataFrame:
    fi = feature_importance.copy().reset_index(drop=True)
    fi["rank"] = range(1, len(fi) + 1)
    fi["model_name"] = best_model.name
    fi.columns = [c.upper() for c in fi.columns]
    return fi


def build_account_scores_frame() -> pd.DataFrame:
    scores = account_scores.copy()
    scores["snapshot_date"] = scores["snapshot_date"].astype(str)
    scores.columns = [c.upper() for c in scores.columns]
    return scores


def load_private_key(path: str = "rsa_key.p8") -> bytes:
    """Load the RSA private key for Snowflake key-pair (JWT) auth.

    This is the same key the app uses, so notebook and app authenticate
    identically. No browser or MFA prompt is involved.
    """
    passphrase = os.environ.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
    with open(path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=passphrase.encode() if passphrase else None,
        )
    return private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )


def publish_to_snowflake() -> None:
    load_env()
    connection = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USERNAME"],
        private_key=load_private_key(),
        role=os.environ.get("SNOWFLAKE_ROLE"),
        warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ["SNOWFLAKE_DATABASE"],
        schema=os.environ["SNOWFLAKE_SCHEMA"],
    )
    try:
        tables = {
            "ML_MODEL_METRICS": build_metrics_frame(),
            "ML_CALIBRATION": build_calibration_frame(),
            "ML_FEATURE_IMPORTANCE": build_feature_importance_frame(),
            "ML_ACCOUNT_SCORES": build_account_scores_frame(),
        }
        for table_name, frame in tables.items():
            write_pandas(
                connection,
                frame,
                table_name,
                auto_create_table=True,
                overwrite=True,
            )
            print(f"Published {len(frame):>5} rows -> {table_name}")
    finally:
        connection.close()


publish_to_snowflake()
print("Snowflake tables refreshed. The app now reads these live tables via /api/ml-scoring.")

Published     8 rows -> ML_MODEL_METRICS
Published    20 rows -> ML_CALIBRATION
Published    40 rows -> ML_FEATURE_IMPORTANCE
Published  3596 rows -> ML_ACCOUNT_SCORES
Snowflake tables refreshed. The app now reads these live tables via /api/ml-scoring.


## 14. How To Interpret Results

A good senior-level readout should focus on business lift, not only statistical metrics.

Recommended executive summary format:

1. **Top-decile lift:** How much better are the top 10% scored accounts than random?
2. **Revenue capture:** What percent of future won revenue is captured by the top-ranked accounts?
3. **Calibration:** Are predicted probabilities reliable enough for routing decisions?
4. **Drivers:** Which product behaviors and GTM signals explain high scores?
5. **Action:** Which accounts should Sales prioritize this week and why?

Production next steps:

- Replace deal `created_date` target with true close/stage transition date
- Add SHAP explanations
- Backtest by quarter
- Monitor data drift and calibration drift
- Retrain monthly
- Expose scores in the Product Usage Analytics page or CRM

## 15. Account-Level Explanations for Sales

Global feature importance tells us what the model uses overall, but Sales needs a more practical answer:

> Why is this specific account ranked high right now?

This section creates account-level reason codes for the highest-priority accounts. These are not causal claims. They are model explanation aids that translate the trained model's strongest signals into Sales-readable reasons.

For each top-ranked account, we produce:

- predicted 90-day win probability
- priority rank
- top model drivers for that account
- a concise Sales-facing explanation
- a recommended sales action

Production note: if `shap` is available, SHAP values are the preferred approach. The fallback below uses a local contribution proxy based on transformed feature values and the deployed model's coefficients or feature importances.

In [36]:
def clean_feature_name(feature_name: str) -> str:
    name = feature_name
    if name.startswith("num__"):
        name = name.replace("num__", "")
    elif name.startswith("cat__"):
        name = name.replace("cat__", "")

    replacements = {
        "event_count__": "event count: ",
        "activity_count__": "sales activity: ",
        "lead_status_count__": "lead status: ",
        "estimated_annual_recurring_revenue": "estimated annual recurring revenue",
        "avg_experience_years": "average user experience years",
        "avg_call_duration_seconds": "average sales call duration",
        "days_since_last_sales_activity": "sales engagement recency",
        "days_since_last_event": "product activity recency",
        "event_momentum_30d": "30-day product usage momentum",
        "total_events_to_date": "total product events to date",
        "active_users_to_date": "active product users to date",
        "distinct_events_to_date": "distinct product events used",
        "total_lead_cost": "total marketing spend on account",
        "marketing_opt_in_rate": "marketing opt-in rate",
        "prior_pipeline_amount": "prior pipeline amount",
        "prior_max_seats": "maximum seats on prior deals",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)

    return name.replace("_", " ").strip()


def driver_theme(feature_name: str) -> str:
    name = feature_name.lower()
    if any(token in name for token in ["event", "workspace", "report", "dashboard", "workflow", "api", "product"]):
        return "Product usage"
    if any(token in name for token in ["activity", "call", "sales", "sdr", "demo", "contract"]):
        return "Sales engagement"
    if any(token in name for token in ["lead", "campaign", "utm", "marketing", "source"]):
        return "Marketing"
    if any(token in name for token in ["revenue", "segment", "industry", "seats", "experience", "job", "users"]):
        return "Company fit"
    if any(token in name for token in ["country", "continent", "geo"]):
        return "Geography"
    return "Other"


def build_local_contribution_matrix(fitted: FittedModel, X: pd.DataFrame) -> tuple[np.ndarray, list[str], str]:
    """Return a local contribution proxy for account-level reason codes.

    Preferred production method is SHAP. This dependency-light fallback works
    for all trained models in this notebook:
    - Logistic regression: transformed feature value * coefficient.
    - Tree models: transformed feature value * global feature importance.

    For tree models, this is a reason-code heuristic rather than exact
    attribution. It is useful for Sales-facing explanations when SHAP is not
    installed.
    """
    pipeline = fitted.pipeline
    transformed = pipeline.named_steps["preprocess"].transform(X)
    transformed = np.asarray(transformed)
    feature_names = get_feature_names(pipeline)
    model = pipeline.named_steps["model"]

    if hasattr(model, "coef_"):
        weights = model.coef_[0]
        method = "coefficient_contribution"
    elif hasattr(model, "feature_importances_"):
        weights = model.feature_importances_
        method = "importance_weighted_signal"
    else:
        weights = np.ones(transformed.shape[1]) / transformed.shape[1]
        method = "unweighted_signal"

    contributions = transformed * weights
    return contributions, feature_names, method


def recommended_sales_action(probability: float, reasons: list[dict[str, object]]) -> str:
    themes = {str(reason["theme"]) for reason in reasons}
    if probability >= 0.40 and "Sales engagement" in themes:
        return "Route to AE for immediate follow-up; reference recent sales engagement."
    if probability >= 0.30 and "Product usage" in themes:
        return "Send product-led expansion outreach tied to active workflows and reports."
    if probability >= 0.20 and "Company fit" in themes:
        return "Prioritize executive or economic-buyer outreach; account fit is strong."
    if probability >= 0.15:
        return "Add to weekly SDR sequence with personalized usage proof."
    return "Monitor and nurture; not a top-priority sales motion yet."


def explain_ranked_accounts(
    fitted: FittedModel,
    scored_accounts: pd.DataFrame,
    X: pd.DataFrame,
    source_rows: pd.DataFrame,
    top_accounts: int = 25,
    reasons_per_account: int = 5,
) -> pd.DataFrame:
    ranked = scored_accounts.sort_values("priority_rank").head(top_accounts).copy()
    ranked_indices = ranked.index.to_list()

    local_X = X.loc[ranked_indices]
    local_source = source_rows.loc[ranked_indices]
    contributions, feature_names, method = build_local_contribution_matrix(fitted, local_X)

    explanation_rows = []
    for row_position, original_index in enumerate(ranked_indices):
        account = ranked.loc[original_index]
        _raw = local_source.loc[original_index]
        row_contributions = contributions[row_position]
        top_feature_indices = np.argsort(row_contributions)[::-1][:reasons_per_account]

        reasons = []
        for feature_index in top_feature_indices:
            raw_feature = feature_names[feature_index]
            readable_feature = clean_feature_name(raw_feature)
            contribution = float(row_contributions[feature_index])
            if contribution <= 0:
                continue
            reasons.append({
                "feature": readable_feature,
                "theme": driver_theme(readable_feature),
                "contribution": contribution,
            })

        reason_text = "; ".join(
            f"{reason['theme']}: {reason['feature']}" for reason in reasons[:3]
        )
        if not reason_text:
            reason_text = "No dominant positive driver; review account context manually."

        explanation_rows.append({
            "snapshot_date": account["snapshot_date"],
            "account_id": account["account_id"],
            "account_name": account["account_name"],
            "industry": account["industry"],
            "segment": account["segment"],
            "estimated_annual_recurring_revenue": account["estimated_annual_recurring_revenue"],
            "priority_rank": account["priority_rank"],
            "predicted_win_probability_90d": account["predicted_win_probability_90d"],
            "actual_won_90d": account["target"],
            "actual_revenue_90d": account["target_revenue_90d"],
            "explanation_method": method,
            "top_reason_1": reasons[0]["feature"] if len(reasons) > 0 else "",
            "top_reason_1_theme": reasons[0]["theme"] if len(reasons) > 0 else "",
            "top_reason_2": reasons[1]["feature"] if len(reasons) > 1 else "",
            "top_reason_2_theme": reasons[1]["theme"] if len(reasons) > 1 else "",
            "top_reason_3": reasons[2]["feature"] if len(reasons) > 2 else "",
            "top_reason_3_theme": reasons[2]["theme"] if len(reasons) > 2 else "",
            "sales_explanation": reason_text,
            "recommended_action": recommended_sales_action(account["predicted_win_probability_90d"], reasons),
        })

    return pd.DataFrame(explanation_rows)


account_level_explanations = explain_ranked_accounts(
    fitted=best_model,
    scored_accounts=account_scores,
    X=X_test,
    source_rows=test_df,
    top_accounts=25,
    reasons_per_account=6,
)

OUTPUT_DIR = Path("ml_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
account_level_explanations.to_csv(OUTPUT_DIR / "account_level_explanations.csv", index=False)

account_level_explanations[[
    "priority_rank",
    "account_name",
    "predicted_win_probability_90d",
    "top_reason_1_theme",
    "top_reason_1",
    "top_reason_2_theme",
    "top_reason_2",
    "top_reason_3_theme",
    "top_reason_3",
    "recommended_action",
]].head(15)


def publish_account_explanations_to_snowflake(frame: pd.DataFrame) -> None:
    """Publish account-level Sales explanations to Snowflake for the app."""
    load_env(".env")
    prepared = frame.copy()
    prepared["snapshot_date"] = prepared["snapshot_date"].astype(str)
    prepared.columns = [c.upper() for c in prepared.columns]

    connection = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USERNAME"],
        private_key=load_private_key("rsa_key.p8"),
        role=os.environ.get("SNOWFLAKE_ROLE"),
        warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ["SNOWFLAKE_DATABASE"],
        schema=os.environ["SNOWFLAKE_SCHEMA"],
    )
    try:
        write_pandas(
            connection,
            prepared,
            "ML_ACCOUNT_EXPLANATIONS",
            auto_create_table=True,
            overwrite=True,
        )
        print(f"Published {len(prepared):>5} rows -> ML_ACCOUNT_EXPLANATIONS")
    finally:
        connection.close()


publish_account_explanations_to_snowflake(account_level_explanations)
print("Account explanations published. The app reads ML_ACCOUNT_EXPLANATIONS via /api/ml-scoring.")

,priority_rank,account_name,predicted_win_probability_90d,top_reason_1_theme,top_reason_1,top_reason_2_theme,top_reason_2,top_reason_3_theme,top_reason_3,recommended_action
0,1,"Town Sports International Holdings, Inc.",0.684614,Product usage,event count: workspace created,Product usage,total product events to date,Product usage,event count: workflow created,Send product-led expansion outreach tied to ac...
1,2,"Town Sports International Holdings, Inc.",0.569455,Product usage,event count: workspace created,Product usage,total product events to date,Product usage,event count: workflow created,Send product-led expansion outreach tied to ac...
2,3,"Town Sports International Holdings, Inc.",0.567830,Product usage,event count: workspace created,Product usage,total product events to date,Product usage,event count: workflow created,Send product-led expansion outreach tied to ac...
3,4,Deltic Timber Corporation,0.541559,Product usage,total product events to date,Product usage,event count: report generated,Product usage,event count: workflow created,Route to AE for immediate follow-up; reference...
4,5,"CommScope Holding Company, Inc.",0.526930,Product usage,event count: workspace created,Product usage,event count: workflow created,Product usage,total product events to date,Send product-led expansion outreach tied to ac...
5,6,"Alliance One International, Inc.",0.505314,Product usage,event count: workspace created,Sales engagement,sales engagement recency,Product usage,event count: workflow created,Route to AE for immediate follow-up; reference...
6,7,"Town Sports International Holdings, Inc.",0.493953,Product usage,event count: workspace created,Product usage,total product events to date,Product usage,event count: workflow created,Send product-led expansion outreach tied to ac...
7,8,"CommScope Holding Company, Inc.",0.475289,Product usage,event count: workspace created,Product usage,event count: workflow created,Product usage,event count: api call made,Send product-led expansion outreach tied to ac...
8,9,"Apricus Biosciences, Inc.",0.447426,Product usage,event count: workspace created,Product usage,event count: workflow created,Product usage,total product events to date,Send product-led expansion outreach tied to ac...
9,10,"Town Sports International Holdings, Inc.",0.433511,Product usage,event count: workspace created,Product usage,total product events to date,Product usage,event count: workflow created,Send product-led expansion outreach tied to ac...


In [ ]:
account_level_explanations